# gru_v5 fine-tuning on Colab (T4): gru_v5a then gru_v5b (warm start from weights/gru_v4b.pt)
Runtime -> Change runtime type -> T4 GPU, then Run all. After a disconnect, just Run all again: each config resumes from its `last.pt` on Drive (`--resume`), and a config whose `train_log.csv` already reached the final epoch or has an `early_stop` row is skipped.

Both: mixture gain augmentation, cmag + MR-STFT (0.5) + negative SI-SDR; v5a: lambda_sisdr 0.2, v5b: lambda_sisdr 0.3; lr 2e-4, 100 epochs, early stop after 8 val points without gain.

Drive layout: dataset at `MyDrive/noise_reduction_challenge/{train,validation}/...`; checkpoints go to `MyDrive/nrc_ckpt/<config_stem>/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
!rm -rf /content/repo && git clone -b exp/dl-v5 --depth 1 https://github.com/pauljh4323/SGSYS_13thTeam_NoiseReductionChallenge.git /content/repo
%cd /content/repo
assert os.path.getsize('weights/gru_v4b.pt') > 1_000_000, 'weights/gru_v4b.pt missing (needed for init_from)'
print('init weights OK:', os.path.getsize('weights/gru_v4b.pt'), 'bytes')

In [ ]:
!pip install -q pesq==0.0.4 pystoi soundfile pyyaml

In [ ]:
!mkdir -p /content/data && cp -rn /content/drive/MyDrive/noise_reduction_challenge /content/data/

In [ ]:
import os, yaml, pandas as pd
from pathlib import Path
CONFIGS = ["configs/dl/gru_v5a.yaml", "configs/dl/gru_v5b.yaml"]
CKPT_ROOT = Path('/content/drive/MyDrive/nrc_ckpt')

def ckpt_dir(cfg): return CKPT_ROOT / Path(cfg).stem

def finished(cfg):
    log = ckpt_dir(cfg) / 'train_log.csv'
    epochs = yaml.safe_load(open(cfg))['train']['epochs']
    if not log.is_file() or os.path.getsize(log) == 0:
        return False
    df = pd.read_csv(log)
    if len(df) == 0:
        return False
    early = 'early_stop' in df and (pd.to_numeric(df.early_stop, errors='coerce') == 1).any()
    return bool(early) or df.epoch.max() >= epochs

for cfg in CONFIGS:
    d = ckpt_dir(cfg)
    if finished(cfg):
        print(f'skip {cfg}: final epoch reached or early-stopped')
        continue
    d.mkdir(parents=True, exist_ok=True)
    !python scripts/train.py --config {cfg} --data-root /content/data/noise_reduction_challenge --ckpt-dir "{d}" --resume --device auto

In [ ]:
for cfg in CONFIGS:
    log = ckpt_dir(cfg) / 'train_log.csv'
    if not log.is_file():
        print(f'{Path(cfg).stem}: no train_log.csv'); continue
    df = pd.read_csv(log).dropna(subset=['val_pesq'])
    if df.empty:
        print(f'{Path(cfg).stem}: no validation yet'); continue
    r = df.loc[df.val_pesq.idxmax()]
    print(f'{Path(cfg).stem}: best epoch {int(r.epoch)}, best val PESQ {r.val_pesq:.4f} (last epoch {int(pd.read_csv(log).epoch.max())})')